# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook guides you through loading, inspecting, and processing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements are referenced by their unique Croissant `@id` identifiers.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Published: {metadata.date_published if hasattr(metadata, 'date_published') else metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. This gives an understanding of the dataset structure before loading any tabular record data.

In [ ]:
# List all available record sets with their @id and name
print("Available record sets:")
for rs in dataset.record_sets:
    print(f"  @id: {rs.id} | name: {getattr(rs, 'name', '[no name]')}")
    # List available fields for each record set
    print("    Fields:")
    for field in rs.fields:
        print(f"      @id: {field.id} | name: {getattr(field, 'name', '[no name]')} | type: {field.data_type}")
    print("")

## 3. Data Extraction
Load records from each record set into a DataFrame for analysis.

**Note:** *We use the record set and field `@id`s identified above to keep references explicit and reproducible.*

In [ ]:
# Prepare the list of record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}

for record_set_id in record_set_ids:
    # Load records using mlcroissant's record_set @id
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Print available DataFrames and preview columns of the first available record set
if dataframes:
    example_record_set = list(dataframes.keys())[0]
    print(f"Loaded {len(dataframes)} record sets. Previewing columns in record set '@id': {example_record_set}")
    print(dataframes[example_record_set].columns.tolist())
    display(dataframes[example_record_set].head())
else:
    print("No tabular record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping. The analysis is based on record fields referenced by their unique `@id`s.

In [ ]:
if dataframes:
    # For demonstration, select the first available record set and identify its numeric fields
    rs_id = example_record_set
    df = dataframes[rs_id]

    # Identify candidate numeric fields by dtype
    numeric_fields = df.select_dtypes(include='number').columns.tolist()

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for analysis: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
        # Filter and normalize
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a likely categorical field, here chosen as the first non-numeric
        group_fields = df.select_dtypes(include='object').columns.tolist()
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"{numeric_field_id}_mean")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No record sets with tabular data for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

*Below, a histogram and a boxplot are displayed for one numeric field, and a bar plot for the first grouping variable, if available.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dataframes and numeric_fields:
    # Numeric field histogram
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.histplot(df[numeric_field_id].dropna(), ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    sns.boxplot(y=df[numeric_field_id], ax=axes[1])
    axes[1].set_title(f"Boxplot of {numeric_field_id}")

    plt.tight_layout()
    plt.show()

    # Bar plot for group_field if available
    if group_fields:
        plt.figure(figsize=(10, 5))
        # Group means
        group_means = df.groupby(group_field)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(f"{group_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
- In this notebook, we've demonstrated how to load and explore a Croissant schema-defined dataset using the `mlcroissant` Python library.
- You learned how to:
  - Inspect the dataset structure and available record sets by their Croissant `@id`.
  - Extract records into DataFrames, filter and normalize numeric fields, and group by categorical variables.
  - Visualize key features using standard Python libraries.

**Next steps:** Consider performing domain-specific analyses such as regression modeling or deeper missing-value diagnostics, referencing Croissant `@id`s throughout to maintain reproducibility and schema traceability.